In [1]:
import pandas as pd
# Import all functions from the required modules
from pop_functions_module import *
#from plot_module import *
print("Successfully loaded population functions")

Successfully loaded population functions


In [2]:
import geopandas as gpd
import os
import glob
# --- Path to folder containing shapefile ---
path_contours = "data/1-processed-data/CONTOURS-IRIS"
# --- Find the .shp file automatically ---
shp_files = glob.glob(os.path.join(path_contours, "*.shp"))
if not shp_files:
    raise FileNotFoundError("No .shp file found in the directory")
shp_path = shp_files[0]  # take the first shapefile
print(f"Using shapefile: {shp_path}")

# --- Read shapefile ---
gdf = gpd.read_file(shp_path)

# --- Inspect columns ---
print("Columns:", gdf.columns.tolist())

target = [
    "940220109", "940220110", "940220111",
    "940220112", "940220113", "940220114"
]

iris_col = "CODE_IRIS"
iris_values = gdf[iris_col].astype(str).str.zfill(9)
iris_set = set(iris_values)

for t in target:
    if t in iris_set:
        print(f"IRIS {t} IS in the shapefile")
    else:
        print(f"IRIS {t} is NOT in the shapefile")

        prefix = t[:5]
        similar = iris_values[iris_values.str.startswith(prefix)].unique().tolist()

        print(f"  Similar IRIS with same commune prefix ({prefix}):")
        print(f"  {similar}\n")

Using shapefile: data/1-processed-data/CONTOURS-IRIS\CONTOURS-IRIS.shp
Columns: ['ID', 'INSEE_COM', 'NOM_COM', 'IRIS', 'CODE_IRIS', 'NOM_IRIS', 'TYP_IRIS', 'geometry']
IRIS 940220109 IS in the shapefile
IRIS 940220110 IS in the shapefile
IRIS 940220111 IS in the shapefile
IRIS 940220112 IS in the shapefile
IRIS 940220113 IS in the shapefile
IRIS 940220114 IS in the shapefile


In [3]:
# Paths to data files
path_contours = "data/1-processed-data/CONTOURS-IRIS"
path_iris = "data/1-processed-data/INSEE/donnees_iris.txt"
path_proj = "data/1-processed-data/INSEE/donnees_projections.txt"
path_depart = "data/1-processed-data/INSEE/num_depart.csv"
path_age_femmes = "data/1-processed-data/INSEE/donnees_age_femmes.csv"
path_age_hommes = "data/1-processed-data/INSEE/donnees_age_hommes.csv"
path_mortalite_hf = "data/1-processed-data/INSEE/donnees_deces_hf.csv"
#path_pop_density = "data/1-processed-data/INSEE/donnees_dens.csv"

# EXPORT : paths and titles for exported data
path = "data/2-output-data"
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"

In [4]:
donnees_geom = geometries(path_iris, path_depart, path_contours)
print("Top 2 rows of donnees_geom:")
print(donnees_geom.head())
age_hf = age_nat(path_age_femmes, path_age_hommes)
print("Top 2 rows of age_hf:")
print(age_hf.head())
perc_hf = decomposition_age(age_hf)
print("All columns of perc_hf:")
print(perc_hf.columns)
print("Top 2 rows of perc_hf:")
print(perc_hf.head())
pd.set_option('display.max_columns', None)  # Ensures all columns are shown
donnees_proj = recense(path_proj, perc_hf)
print("Top 2 rows of donnees_proj:")
print(donnees_proj.head())
donnees_insee = desagreg(donnees_geom, donnees_proj)
print("Top 2 rows of donnees_insee:")
print(donnees_insee.head())
donnees_dens = dens(donnees_insee)
print("Top 2 rows of donnees_dens:")
print(donnees_dens.head())
donnees_finales = mortalite(donnees_dens, path_mortalite_hf)
print("Top 2 rows of donnees_finales:")
print(donnees_finales.head())
donnees_filtrees = create_donnees_shp(donnees_finales)

The INSEE data is extracted at the IRIS level.
Top 2 rows of donnees_geom:
  dep_cod           dep_name  region_cod                 region_name com_cod  \
0      67           Bas-Rhin        44.0                   Grand Est   67043   
1      13   Bouches-du-Rhône        93.0  Provence-Alpes-Côte d’Azur   13202   
2      56           Morbihan        53.0                    Bretagne   56185   
3      93  Seine-Saint-Denis        11.0               Ile-de-France   93063   
4      94       Val-de-Marne        11.0               Ile-de-France   94048   

                      com_name   iris_cod           iris_name  \
0                    Bischheim  670430101              Annexe   
1  Marseille 2e Arrondissement  132020101               Arenc   
2                       Quéven  561850101  B.A.N. Lann Bihoué   
3                  Romainville  930630101            Bas Pays   
4             Marolles-en-Brie  940480101  Bois de Notre-Dame   

   POP_2019_IRIS_TotAge                              

C:\Users\aysharma\miniconda3\envs\airquality\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


The population predictions between 2019 and 2050 have been calculated at the IRIS level.
Top 2 rows of donnees_insee:
  dep_cod  dep_name  region_cod region_name com_cod   com_name   iris_cod  \
0      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
1      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
2      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
3      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
4      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   

  iris_name  POP_2019_IRIS_TotAge  \
0    Annexe                  99.0   
1    Annexe                  99.0   
2    Annexe                  99.0   
3    Annexe                  99.0   
4    Annexe                  99.0   

                                            geometry       aire_m2      ZONE  \
0  POLYGON ((1052346.7 6848413, 1052502.3 6848474...  1.623617e+06  Bas-Rhin   
1  POLYGON ((1052346.7 6848413, 1052

C:\Users\aysharma\miniconda3\envs\airquality\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
C:\Users\aysharma\miniconda3\envs\airquality\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
C:\Users\aysharma\miniconda3\envs\airquality\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

Mortality disaggregation from 2019 to 2050 has been calculated at the IRIS level.
Top 2 rows of donnees_finales:
  dep_cod  dep_name  region_cod region_name com_cod   com_name   iris_cod  \
0      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
1      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
2      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
3      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   
4      67  Bas-Rhin        44.0   Grand Est   67043  Bischheim  670430101   

  iris_name  POP_2019_IRIS_TotAge  \
0    Annexe                  99.0   
1    Annexe                  99.0   
2    Annexe                  99.0   
3    Annexe                  99.0   
4    Annexe                  99.0   

                                            geometry       aire_m2      ZONE  \
0  POLYGON ((1052346.7 6848413, 1052502.3 6848474...  1.623617e+06  Bas-Rhin   
1  POLYGON ((1052346.7 6848413, 1052502.3

In [5]:
print(donnees_finales[['age', 'MORT_2019_IRIS']].groupby('age').sum())
# After processing
age_numeric = pd.to_numeric(donnees_finales["age"], errors="coerce")
for yr in [2019, 2030, 2050]:
    mort_col = f"MORT_{yr}_IRIS"
    deaths_nat = donnees_finales.loc[age_numeric >= 30, mort_col].sum()
    print(f"France, deaths age 30+ in {yr}: {int(deaths_nat):,}")


     MORT_2019_IRIS
age                
0       2410.801177
1        386.255757
10        54.552045
11        42.324863
12        59.254808
..              ...
95     17779.321187
96     15199.116615
97     12734.080879
98     10512.061953
99     28661.202117

[100 rows x 1 columns]
France, deaths age 30+ in 2019: 598,008
France, deaths age 30+ in 2030: 693,893
France, deaths age 30+ in 2050: 882,564


In [6]:
donnees_filtrees = create_donnees_shp(donnees_finales)

donnees_merged = donnees_filtrees.copy()
donnees_merged["age"] = pd.to_numeric(donnees_merged["age"], errors="coerce")

cols_to_sum = ["pop2019", "pop2030", "pop2050", "mort2019", "mort2030", "mort2050"]
for col in cols_to_sum:
    donnees_merged[col] = pd.to_numeric(donnees_merged[col], errors="coerce")

donnees_merged = donnees_merged[(donnees_merged["age"] >= 30) & (donnees_merged["age"] <= 99)]
sum_values = donnees_merged[cols_to_sum].sum(skipna=True)

print(sum_values)


pop2019     4.138219e+07
pop2030     4.319732e+07
pop2050     4.516190e+07
mort2019    5.980088e+05
mort2030    6.938935e+05
mort2050    8.825646e+05
dtype: float64


In [10]:
donnees_shp = export_pollution(donnees_filtrees, donnees_geom)
gdf = gpd.GeoDataFrame(donnees_shp, geometry=donnees_geom.geometry)

# Convert other dataframes to GeoDataFrames
donnees_shp_1 = donnees_filtrees.query("age >= 0 & age <= 55").loc[:, [
                                                                          "iriscod", "irisname", "comcod", "comname",
                                                                          "depcod", "depname", "regcod", "regname",
                                                                          "age",
                                                                          "pop2019", "pop2030", "pop2050", "mort2019",
                                                                          "mort2030", "mort2050"
                                                                      ]]
gdf_1 = gpd.GeoDataFrame(donnees_shp_1, geometry=donnees_geom.geometry)

donnees_shp_2 = donnees_filtrees.query("age >= 56 & age <= 87").loc[:, [
                                                                           "iriscod", "irisname", "comcod", "comname",
                                                                           "depcod", "depname", "regcod", "regname",
                                                                           "age",
                                                                           "pop2019", "pop2030", "pop2050", "mort2019",
                                                                           "mort2030", "mort2050"
                                                                       ]]
gdf_2 = gpd.GeoDataFrame(donnees_shp_2, geometry=donnees_geom.geometry)

donnees_shp_3 = donnees_filtrees.query("age >= 88 & age <= 99").loc[:, [
                                                                           "iriscod", "irisname", "comcod", "comname",
                                                                           "depcod", "depname", "regcod", "regname",
                                                                           "age",
                                                                           "pop2019", "pop2030", "pop2050", "mort2019",
                                                                           "mort2030", "mort2050"
                                                                       ]]
gdf_3 = gpd.GeoDataFrame(donnees_shp_3, geometry=donnees_geom.geometry)

export_data_shp(gdf, path_fichier_shp, title_shp)
export_data_shp(gdf_1, path_fichier_shp_1, title_shp_1)
export_data_shp(gdf_2, path_fichier_shp_2, title_shp_2)
export_data_shp(gdf_3, path_fichier_shp_3, title_shp_3)

# Functions for exportation
title_csv = "my_data_file"
#export_data_csv(donnees_finales, path, title_csv)
#export_data_shp(donnees_indic, path_fichier_shp, title_shp)
#donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
#print(donnees_exportees.head())


Shapefile written to: data/2-output-data/donnees_shp\donnees_insee_iris.shp
Shapefile written to: data/2-output-data/donnees_shp_1\donnees_insee_iris_toutage_1.shp
Shapefile written to: data/2-output-data/donnees_shp_2\donnees_insee_iris_toutage_2.shp
Shapefile written to: data/2-output-data/donnees_shp_3\donnees_insee_iris_toutage_3.shp


In [ ]:
# Functions to test the disaggregation and downscaling
annee = 2030
dep_num = "75"
iris_num = "010040102"
n = 35
national_vivant_test(donnees_finales, annee)
national_mort_test(donnees_finales, annee)
departemental_test(donnees_finales, dep_num, annee)
iris_test(donnees_finales)
dep_vs_iris_test(donnees_finales)
pyramide_iris(donnees_insee, iris_num, annee)
pyramide_dep(donnees_insee, dep_num, annee)

In [ ]:
## Function to plot age pyramids of the whole population for multiple years side-by-side
def pyramide_population_3years(donnees_insee, years=(2019, 2030, 2050), y_tick_step=10):
    pop_cols = {y: f"POP_{y}_IRIS" for y in years}
    missing = [c for c in pop_cols.values() if c not in donnees_insee.columns]
    if missing:
        raise KeyError(f"Missing population columns in donnees_insee: {missing}")

    df = donnees_insee.loc[:, ["age", *pop_cols.values()]].copy()
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df = df.dropna(subset=["age"])
    df["age"] = df["age"].astype(int)

    ages = sorted(df["age"].unique().tolist())

    series_by_year = {}
    for y, c in pop_cols.items():
        s = df.groupby("age", as_index=True)[c].sum()
        series_by_year[y] = s.reindex(ages, fill_value=0.0)

    tick_ages = [a for a in ages if (a % y_tick_step) == 0]
    if ages and ages[-1] not in tick_ages:
        tick_ages.append(ages[-1])

    colors = {2019: "#1f77b4", 2030: "#ff7f0e", 2050: "#2ca02c"}

    fig, axes = plt.subplots(nrows=1, ncols=len(years), figsize=(14.5, 5.6), sharey=True)
    if len(years) == 1:
        axes = [axes]

    y_max = max(float(series_by_year[y].max()) for y in years) if ages else 0.0
    x_pad = max(1.0, y_max * 0.06)

    for ax, y in zip(axes, years):
        ax.bar(
            ages,
            series_by_year[y].values,
            width=0.9,
            color=colors.get(y, None),
            alpha=0.9,
        )
        ax.set_title(str(y))
        ax.set_xticks(tick_ages)
        ax.set_xticklabels([str(a) for a in tick_ages], rotation=0)
        ax.set_xlim(min(ages) - 1, max(ages) + 1)
        ax.set_ylim(0, y_max + x_pad)
        ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.4)
        ax.set_xlabel("Age (years)")

    axes[0].set_ylabel("Population")
    fig.suptitle("Population by Age", y=0.98)
    fig.tight_layout()
    plt.show()

In [ ]:
# Age-pyaramide plots for whole population for 3 years
pyramide_population_3years(donnees_insee, years=(2019, 2030, 2050), y_tick_step=10)

In [ ]:
#Gender-pyaramide plots for 3 years
path_age_femmes = "data/1-processed-data/INSEE/donnees_age_femmes.csv"
path_age_hommes = "data/1-processed-data/INSEE/donnees_age_hommes.csv"
pyramide_population_genre_3years(path_age_femmes, path_age_hommes, years=(2019, 2030, 2050), y_tick_step=10)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
# Show density (KDE) plot of mortality distribution for 2019, 2030, 2050
years = [2019, 2030, 2050]
mort_cols = ["mort2019", "mort2030", "mort2050"]
colors = {2019: "#1f77b4", 2030: "#ff7f0e", 2050: "#2ca02c"}

gdf_maps = donnees_exportees.copy()
def _clean_series(s):
    s = s.astype(float)
    s = s.replace([float("inf"), float("-inf")], float("nan")).dropna()
    return s
def _kde_plot(ax, s, label, color, bw_method="scott"):
    s = _clean_series(s)
    if len(s) < 5:
        return
    kde = gaussian_kde(s.values, bw_method=bw_method)
    x0, x1 = s.quantile(0.01), s.quantile(0.99)
    if x0 == x1:
        x0, x1 = float(s.min()), float(s.max())
        if x0 == x1:
            return
    pad = (x1 - x0) * 0.05
    xs = [x0 - pad + i * (x1 - x0 + 2 * pad) / 399 for i in range(400)]
    ys = kde(xs)
    ax.plot(xs, ys, color=color, linewidth=2.2, label=label)
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for yr, col in zip(years, mort_cols):
    _kde_plot(ax, gdf_maps[col], label=str(yr), color=colors[yr])

ax.set_title("Mortality Distribution: Density (KDE) (2019, 2030, 2050)", fontsize=14)
ax.set_xlabel("Mortality")
ax.set_ylabel("Density")
ax.grid(True, alpha=0.25)
ax.legend(title="Year", frameon=False)
plt.show()

In [ ]:
#Code to classify data into Urban vs Rural
# Step 1: Aggregate IRIS-level data to commune level (by "comcod")
commune_level_data = (
    donnees_exportees.groupby("comcod")
    .agg(
        pop2019=("pop2019", "sum"),
        pop2030=("pop2030", "sum"),
        pop2050=("pop2019", "sum"),
        area=("geometry", lambda x: x.union_all().area),  # Compute total area by unioning geometries
        geometry=("geometry", lambda x: x.union_all())  # Combine geometries to create commune-level geometry
    )
    .reset_index()
)

# Step 2: Calculate population density at the commune level
commune_level_data["pop_density"] = commune_level_data["pop2030"] / (commune_level_data["area"] / 1e6)

# Debugging: Display the first few rows of commune-level data
print("Head of commune-level data:")
print(commune_level_data.head())

# Step 3: Classify communes into "Urban" or "Rural" clusters
def classify_area_clusters(row):
    if row["pop_density"] > 1500:
        if row["pop2019"] >= 50000 or (5000 <= row["pop2019"] < 50000):
            return "Urban"  # Merge all Dense Urban levels into Urban
    elif row["pop_density"] > 300 and row["pop2019"] >= 5000:
        return "Urban"  # Include Semi-Dense Urban in Urban category
    else:
        return "Rural"  # Merge Rural and Other together

# Apply classification to the commune-level data
commune_level_data["area_cluster"] = commune_level_data.apply(classify_area_clusters, axis=1)

# Step 4: Debugging - Count clusters after classification
print(f"Clusters at the commune level:\n{commune_level_data['area_cluster'].value_counts()}")

# Step 5: Disaggregate classification back to the IRIS level
# Merge commune-level results (area_cluster, pop_density) back to the IRIS-level data
donnees_exportees_transformed = donnees_exportees.merge(
    commune_level_data[["comcod", "area_cluster", "pop_density"]],
    on="comcod",  # Match by the commune code
    how="left"  # Retain all IRIS-level data
)

# Step 6: Debugging - Display the first few rows of the disaggregated data
print("Head of IRIS-level data with disaggregated classification:")
print(donnees_exportees_transformed.head())